# Fine-tune BGE-M3 trên Luật BHXH/BLLĐ — v5 Sprint 2 Phase 2

Notebook duy nhất chạy toàn bộ quy trình fine-tune trên Colab Pro.

**Yêu cầu trước khi mở notebook**:
1. Colab Pro với GPU A100 hoặc V100 (Runtime → Change runtime type → GPU class: Premium).
2. Google Drive đã mount (cell 2 sẽ làm).
3. Repo `legal-graph-kb` đã push lên GitHub với branch chứa:
   - `data/finetune-bge/qa_pairs_v1.jsonl` (Phase 1 output)
   - `data/eval/questions_50_dev.json` (Phase 0b output)
   - `data/graph/processed/merged_graph.json` (B4 output)

**Workflow**:
1. Setup + clone repo
2. Style spot-check (30 synthetic vs 30 dev) — abort nếu drift quá lớn
3. Load BGE-M3 + apply LoRA
4. Training
5. Dev recall@K evaluation
6. Export adapter to Drive

**Output adapter** sẽ ở `/content/drive/MyDrive/bge-m3-bhxh-lora/`. User tải về local `models/bge-m3-bhxh-lora/`, sau đó Phase 3:
```bash
python -m offline.embed --adapter-path models/bge-m3-bhxh-lora \
    --output data/graph/processed/embeddings_tuned.parquet --force
python -m offline.load_neo4j --apply-schema --embeddings-only \
    --embeddings data/graph/processed/embeddings_tuned.parquet \
    --embed-prop embedding_tuned
```

## 1. Install + setup

In [ ]:
# ===== USER CONFIG (chỉnh trước khi chạy) =====
REPO_URL  = "https://github.com/<YOUR_USER>/legal-graph-kb.git"  # CHỈNH
BRANCH    = "main"

# Hyperparameters (plan §3 Phase 2 defaults)
SEED            = 42
LORA_R          = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.1
BATCH_SIZE      = 8         # V100 16GB; A100 40GB có thể dùng 16-32
GRAD_ACCUM      = 4
LEARNING_RATE   = 2e-5
NUM_EPOCHS      = 2
WARMUP_RATIO    = 0.1
MAX_HARD_NEGS   = 5         # cap mỗi training row
EVAL_TOP_K      = 10        # recall@K trên dev

# Drive output location (notebook tự tạo nếu chưa có)
DRIVE_ROOT      = "/content/drive/MyDrive"
ADAPTER_NAME    = "bge-m3-bhxh-lora"

In [ ]:
# Install deps. sentence-transformers ≥3.0 cần peft ≥0.7
!pip install -q sentence-transformers==3.2.1 peft==0.13.2 accelerate==1.0.1 datasets==3.0.1

In [ ]:
# Mount Drive (persists adapter qua Colab session timeout)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo (fresh mỗi session để không dùng cache cũ)
!rm -rf /content/repo
!git clone -b {BRANCH} {REPO_URL} /content/repo
%cd /content/repo
!git log -1 --oneline

In [ ]:
# GPU sanity check
import torch
assert torch.cuda.is_available(), "GPU không available — đổi runtime sang GPU Premium"
dev_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {dev_name}  |  VRAM: {vram_gb:.1f} GB")
if vram_gb < 15:
    print("WARN: VRAM < 15 GB — giảm BATCH_SIZE xuống 4 hoặc đổi runtime.")

## 2. Verify data + style spot-check

Đây là gate đầu tiên: 30 synthetic Q vs 30 dev Q. Review thủ công nhanh — nếu thấy style synthetic quá khác dev (vd dev luôn có context cá nhân nhưng synthetic toàn câu cộc lốc) → **dừng**, sửa prompt `prompts/offline/synthetic_query_gen.md`, regenerate Phase 1, push lại branch, re-run notebook.

In [ ]:
import json, random
from pathlib import Path

SYNTH_FILE = Path('data/finetune-bge/qa_pairs_v1.jsonl')
DEV_FILE   = Path('data/eval/questions_50_dev.json')
GRAPH_FILE = Path('data/graph/processed/merged_graph.json')
LOCK_FILE  = Path('data/eval/eval_split_hashes.json')

for p in (SYNTH_FILE, DEV_FILE, GRAPH_FILE, LOCK_FILE):
    assert p.exists(), f'Thiếu {p}. Đã push branch đầy đủ chưa?'

synthetic = [json.loads(l) for l in SYNTH_FILE.read_text(encoding='utf-8').splitlines() if l.strip()]
dev = json.loads(DEV_FILE.read_text(encoding='utf-8'))
graph = json.loads(GRAPH_FILE.read_text(encoding='utf-8'))
lock = json.loads(LOCK_FILE.read_text(encoding='utf-8'))

print(f'synthetic rows : {len(synthetic)}')
print(f'dev questions  : {len(dev)}')
print(f'KG clauses     : {len(graph["nodes"]["Clause"])}')
print(f'dev hash lock  : {lock["dev_sha256"][:16]}... (must not drift)')

In [ ]:
# Style spot-check — render side-by-side
rng = random.Random(SEED)
synth_sample = rng.sample(synthetic, k=min(30, len(synthetic)))
dev_sample   = rng.sample(dev, k=min(30, len(dev)))

import statistics
def word_count(s): return len(s.split())
wc_synth = [word_count(r['query']) for r in synth_sample]
wc_dev   = [word_count(d['question']) for d in dev_sample]
print(f"Word count — synthetic: mean={statistics.mean(wc_synth):.1f} median={statistics.median(wc_synth)}")
print(f"Word count — dev      : mean={statistics.mean(wc_dev):.1f}   median={statistics.median(wc_dev)}")
print("\n=== STYLE SAMPLE (side-by-side) ===\n")
for i, (s, d) in enumerate(zip(synth_sample, dev_sample), 1):
    print(f"[{i:>2}] SYNTH : {s['query']}")
    print(f"     DEV   : {d['question'][:200]}{'...' if len(d['question'])>200 else ''}")
    print()
print('--- REVIEW ---')
print('Nếu style synthetic xa dev (vd: ngắn cộc lốc vs narrative dài) → DỪNG, sửa prompt + regen Phase 1.')

## 3. Build training dataset

Multi-positive được flatten: mỗi `(query, pos_i)` thành 1 row, kèm tối đa `MAX_HARD_NEGS` hard negative. `MultipleNegativesRankingLoss` xử lý in-batch negative + explicit hard negative tự động.

In [ ]:
from sentence_transformers import InputExample

train_examples = []
for row in synthetic:
    negs = row.get('neg', [])[:MAX_HARD_NEGS]
    for pos_text in row['pos']:
        # MNRL: texts=[anchor, positive, neg1, neg2, ...]
        train_examples.append(InputExample(texts=[row['query'], pos_text] + negs))

print(f'Training rows (after multi-positive flatten): {len(train_examples)}')
rng.shuffle(train_examples)
print('Sample row [0]:')
ex = train_examples[0]
print(f'  anchor : {ex.texts[0]}')
print(f'  pos    : {ex.texts[1][:180]}...')
for i, n in enumerate(ex.texts[2:], 1):
    print(f'  neg {i}  : {n[:120]}...')

## 4. Load BGE-M3 + apply LoRA

In [ ]:
from sentence_transformers import SentenceTransformer, losses
from peft import LoraConfig, get_peft_model, TaskType

model = SentenceTransformer('BAAI/bge-m3', device='cuda')
print(f'BGE-M3 loaded. embedding dim = {model.get_sentence_embedding_dimension()}')

underlying = model._modules['0'].auto_model
lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['query', 'key', 'value', 'dense'],
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type=TaskType.FEATURE_EXTRACTION,
)
model._modules['0'].auto_model = get_peft_model(underlying, lora_cfg)
model._modules['0'].auto_model.print_trainable_parameters()

## 5. Training

In [ ]:
import math
from torch.utils.data import DataLoader

train_loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE)
train_loss = losses.MultipleNegativesRankingLoss(model)

steps_per_epoch = math.ceil(len(train_loader))
warmup_steps = int(steps_per_epoch * NUM_EPOCHS * WARMUP_RATIO)
print(f'steps/epoch: {steps_per_epoch}, epochs: {NUM_EPOCHS}, warmup: {warmup_steps}')

CKPT_TMP = '/content/bge_m3_lora_ckpt'
model.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=NUM_EPOCHS,
    warmup_steps=warmup_steps,
    optimizer_params={'lr': LEARNING_RATE},
    output_path=CKPT_TMP,
    save_best_model=False,
    use_amp=True,
    show_progress_bar=True,
)
print('\nTraining finished.')

## 6. Dev recall@K evaluation

Encode all KG clauses + dev questions với model đã fine-tune. Tính recall@K bằng cách so gold_articles (article-level) với top-K retrieved articles.

Đây không phải replacement cho `eval_core/metrics.py` — chỉ là sanity check trong Colab. Real metric chạy ở local sau Phase 3.

In [ ]:
import re
import numpy as np

# In-corpus law codes (chỉ 3 luật như plan §11)
IN_CORPUS_CODES = {'41/2024/QH15', '45/2019/QH14', '58/2014/QH13'}
CODE_TO_LAW = {'41/2024/QH15': 'L41_2024', '45/2019/QH14': 'L45_2019', '58/2014/QH13': 'L58_2014'}
_RE_CODE = re.compile(r'\d+/\d{4}/(?:QH\d+|N[ĐD]-CP|NQ-CP|TT-[A-Z]+|CP|TTg)')
_RE_ART = re.compile(r'\bĐiều\s+(\d+[a-z]?)', re.IGNORECASE)

def parse_gold(raw):
    if not raw: return set()
    if isinstance(raw, list): raw = '\n'.join(str(x) for x in raw)
    out = set()
    for line in raw.split('\n'):
        codes = _RE_CODE.findall(line)
        arts  = _RE_ART.findall(line)
        for c in codes:
            if c not in IN_CORPUS_CODES:
                continue
            for a in arts:
                out.add(f'{CODE_TO_LAW[c]}.A{a}')
    return out

# Build clause text inventory matching offline/embed.py:collect_units
art_by_id = {a['id']: a for a in graph['nodes']['Article']}
clause_units = []
for n in graph['nodes']['Clause']:
    text = (n.get('text') or '').strip()
    if not text: continue
    art = art_by_id.get(n.get('article_id',''), {})
    art_num = str(art.get('number','')).strip()
    art_title = (art.get('title') or '').strip()
    if art_num and art_title:
        embed_text = f'Điều {art_num}. {art_title}\nKhoản {n["number"]}: {text}'
    else:
        embed_text = text
    clause_units.append({
        'id': n['id'], 'article_id': n.get('article_id',''),
        'embed_text': embed_text,
    })

print(f'Embedding {len(clause_units)} clauses...')
clause_embs = model.encode(
    [u['embed_text'] for u in clause_units],
    batch_size=BATCH_SIZE, normalize_embeddings=True,
    show_progress_bar=True, convert_to_numpy=True,
)
print(f'Encoded shape: {clause_embs.shape}')

In [ ]:
# Eval recall@K on dev (in-corpus subset only — OOC không thể retrieve)
in_corpus_dev = [d for d in dev if parse_gold(d.get('gold_citations_raw'))]
print(f'Dev in-corpus questions: {len(in_corpus_dev)} / {len(dev)}')

if not in_corpus_dev:
    print('WARN: no in-corpus dev questions — skip eval.')
else:
    queries = [d['question'] for d in in_corpus_dev]
    q_embs = model.encode(queries, batch_size=BATCH_SIZE, normalize_embeddings=True,
                          show_progress_bar=False, convert_to_numpy=True)
    sims = q_embs @ clause_embs.T  # (n_q, n_clauses)
    top_k_idx = np.argsort(-sims, axis=1)[:, :EVAL_TOP_K * 3]  # ample then dedupe by article
    recalls = []
    for qi, d in enumerate(in_corpus_dev):
        gold = parse_gold(d.get('gold_citations_raw'))
        if not gold:
            continue
        retrieved_articles = []
        seen = set()
        for j in top_k_idx[qi]:
            aid = clause_units[j]['article_id']
            if aid and aid not in seen:
                seen.add(aid)
                retrieved_articles.append(aid)
            if len(retrieved_articles) >= EVAL_TOP_K:
                break
        hit = len(gold & set(retrieved_articles))
        recalls.append(hit / len(gold))
    print(f'\n=== Dev evaluation (n={len(recalls)}, in-corpus only) ===')
    print(f'  recall@{EVAL_TOP_K}_dev (macro) = {np.mean(recalls):.4f}')
    print(f'  recall@{EVAL_TOP_K}_dev (median) = {np.median(recalls):.4f}')
    print(f'\n(Compare baseline Sprint 1 retrieval recall@10 in-corpus from probe — see experiment README.)')

## 7. Save adapter + export

Adapter saved to Drive — survives Colab disconnect. User tải về local `models/bge-m3-bhxh-lora/`.

In [ ]:
from pathlib import Path
import json, datetime, shutil

ADAPTER_OUT = Path(DRIVE_ROOT) / ADAPTER_NAME
ADAPTER_OUT.mkdir(parents=True, exist_ok=True)

# Save the PEFT adapter (config + safetensors)
model._modules['0'].auto_model.save_pretrained(str(ADAPTER_OUT))

# Training metadata for reproducibility
meta = {
    'script_version': 'v5-sprint2-phase2-notebook-1',
    'colab_runtime': dev_name,
    'vram_gb': round(vram_gb, 1),
    'base_model': 'BAAI/bge-m3',
    'lora': {'r': LORA_R, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT,
             'target_modules': ['query','key','value','dense']},
    'training': {'batch_size': BATCH_SIZE, 'grad_accum': GRAD_ACCUM,
                 'lr': LEARNING_RATE, 'epochs': NUM_EPOCHS,
                 'warmup_ratio': WARMUP_RATIO, 'max_hard_negs': MAX_HARD_NEGS},
    'data': {'synthetic_rows': len(synthetic),
             'training_examples_after_flatten': len(train_examples),
             'dev_eval_n': len(in_corpus_dev) if 'in_corpus_dev' in dir() else None,
             'dev_recall_at_k': float(np.mean(recalls)) if 'recalls' in dir() and recalls else None,
             'eval_top_k': EVAL_TOP_K},
    'eval_split_lock': lock,
    'finished_at_utc': datetime.datetime.utcnow().isoformat() + 'Z',
}
(ADAPTER_OUT / 'colab_session_metadata.json').write_text(
    json.dumps(meta, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

# Zip for quick download via Drive web UI
zip_base = str(ADAPTER_OUT) + '_zip'
shutil.make_archive(zip_base, 'zip', str(ADAPTER_OUT))
print(f'\nAdapter saved to Drive:\n  dir: {ADAPTER_OUT}\n  zip: {zip_base}.zip')
print(f'\nFile list:')
for f in ADAPTER_OUT.iterdir():
    print(f'  {f.name}  ({f.stat().st_size/1024:.1f} KB)' if f.is_file() else f'  {f.name}/')

## 8. Tải về local + Phase 3

Trên local:

1. Tải `bge-m3-bhxh-lora_zip.zip` từ Drive về local.
2. Giải nén vào `models/bge-m3-bhxh-lora/` (tạo folder nếu chưa có).
3. Verify:
   ```bash
   ls models/bge-m3-bhxh-lora/
   # phải có: adapter_config.json, adapter_model.safetensors, colab_session_metadata.json
   ```
4. Encode corpus với adapter:
   ```bash
   python -m offline.embed --adapter-path models/bge-m3-bhxh-lora \
       --output data/graph/processed/embeddings_tuned.parquet --force
   ```
5. Load tuned vectors vào Neo4j (additive, không phá `clause_vec` cũ):
   ```bash
   python -m offline.load_neo4j --apply-schema --embeddings-only \
       --embeddings data/graph/processed/embeddings_tuned.parquet \
       --embed-prop embedding_tuned
   ```
6. Setup runtime env var để Sprint 2 arm dùng tuned model:
   ```bash
   # .env (hoặc shell)
   BGE_M3_ADAPTER_PATH=models/bge-m3-bhxh-lora
   V5_DENSE_INDEX=clause_vec_tuned
   V5_RERANKER_MODEL=BAAI/bge-reranker-base
   ```
7. Chạy Phase 4 experiment: `python -m eval_core all experiments/04_v5_sprint2_m2`.